La API CEMS-GloFAS Historical nos proporciona datos históricos sobre el caudal de los ríos

Las variables que tenemos son:
- valid_time: La fecha del registro (formato diario). Es tu eje temporal.
- latitude / longitude: Coordenadas geográficas. GloFAS usa una rejilla de 0.1° (~11 km).
- caudal_m3s (originalmente dis24): Caudal del río. Es el volumen de agua que pasa por ese punto en $m^3/s$. Es tu variable principal de respuesta.
- ro (Runoff): Escorrentía. Es la lámina de agua que fluye sobre la superficie del terreno hacia los cauces. Se mide en metros ($m$). Es el mejor predictor a corto plazo de una crecida.
- sd (Snow Depth): Equivalente de agua de la nieve. Indica cuánta agua resultaría si se derritiera toda la nieve acumulada en ese punto. Se mide en metros ($m$). Fundamental si tu zona de estudio tiene montañas.
- surface: Humedad en la capa superior del suelo (pocos centímetros). Reacciona muy rápido a la lluvia reciente. Es útil para predecir inundaciones repentinas (flash floods). De 0 a 1
- swir: Humedad en la zona radicular (donde llegan las raíces de las plantas). Representa la reserva de agua profunda. Si esta zona está cerca de 1, el suelo ya no puede absorber más agua y casi toda la lluvia se convertirá en escorrentía (ro), aumentando drásticamente el riesgo de inundación. De 0 a 1
- rootZone: Volumen de agua en la zona radicular. $m^3/m^3$ metro cúbico de agua por metro cúbico de suelo

In [1]:
# Instalación de las librerías necesarias
%pip install cdsapi xarray h5netcdf pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
# Configuramos el mensajero para que vaya al portal de Emergencias
with open(os.path.expanduser('~/.cdsapirc'), 'w') as f:
    f.write('url: https://ewds.climate.copernicus.eu/api\n')
    f.write('key: b1c8ed63-d58a-4ca0-b6a8-0068ce339221')

In [3]:
import cdsapi

client = cdsapi.Client()

dataset = "cems-glofas-historical"
request = {
    "system_version": ["version_4_0"],
    "hydrological_model": ["lisflood"],
    "product_type": ["consolidated"],
    "variable": [
        "river_discharge_in_the_last_24_hours",
        "runoff_water_equivalent",
        "snow_depth_water_equivalent",
        "soil_wetness_index"
    ],
    "hyear": ["2025"],
    "hmonth": ["03"],
    "hday": [f"{i:02d}" for i in range(1, 32)], # Genera del 01 al 31
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [43.51, -4.85, 42.76, -3.15]
}

target_nc = "caudal_completo_raw.nc"

print("Iniciando descarga desde EWDS...")
client.retrieve(dataset, request).download(target_nc)
print(f"Descarga terminada: {target_nc}")

Iniciando descarga desde EWDS...


2026-05-09 18:16:16,050 INFO [2024-02-01T00:00:00] Please note that accessing this dataset via CDS for time-critical operation is not advised or supported
2026-05-09 18:16:16,052 INFO [2024-02-01T00:00:00] Please note we suggest checking the list of known issues on the GloFAS wiki
[here](https://confluence.ecmwf.int/display/CEMS/GloFAS+-+Known+Issues)
before downloading the dataset.
2026-05-09 18:16:16,053 INFO Request ID is 10654e3e-1306-471c-aaa0-4f1da9b46e80
2026-05-09 18:16:16,237 INFO status has been updated to accepted
2026-05-09 18:16:32,758 INFO status has been updated to running
2026-05-09 18:17:35,085 INFO status has been updated to successful
                                                                                      

Descarga terminada: caudal_completo_raw.nc


In [4]:
import xarray as xr
import pandas as pd
import os

input_file = "caudal_completo_raw.nc"
output_csv = "copernicus_caudales_historicos.csv"

try:
    print("Iniciando conversión a CSV...")
    # Abrimos con el motor h5netcdf (asegúrate de tenerlo instalado)
    with xr.open_dataset(input_file, engine='h5netcdf') as ds:
        
        # 1. Convertir a DataFrame y resetear índices
        df = ds.to_dataframe().reset_index()
        
        # 2. Identificar la columna de caudal (suele ser 'dis24')
        # La renombramos para que sea más clara en tu modelo de IA
        col_caudal = [c for c in df.columns if 'dis' in c or 'river' in c]
        if col_caudal:
            df = df.rename(columns={col_caudal[0]: 'caudal_m3s'})
            
            # 3. Limpieza crítica: GloFAS devuelve NaN en píxeles donde no hay cauce.
            # Los eliminamos para quedarnos solo con la red fluvial real.
            df = df.dropna(subset=['caudal_m3s'])
        
        # 4. Guardar resultado
        df.to_csv(output_csv, index=False)
        
    print(f"--- ¡CONVERSIÓN EXITOSA! ---")
    print(f"Archivo guardado: {output_csv}")
    print(f"Número de puntos de río encontrados: {len(df)}")
    display(df.head())

except Exception as e:
    print(f"Error al procesar el archivo: {e}")

Iniciando conversión a CSV...
--- ¡CONVERSIÓN EXITOSA! ---
Archivo guardado: copernicus_caudales_historicos.csv
Número de puntos de río encontrados: 14632


,valid_time,latitude,longitude,rowe,sd,swir,caudal_m3s,surface,rootZone
18,2025-03-02,43.475,356.075,0.653252,0.0,0.821021,0.15625,0.0,0.0
19,2025-03-02,43.475,356.125,0.560638,0.0,0.828020,0.34375,0.0,0.0
20,2025-03-02,43.475,356.175,0.399582,0.0,0.823460,0.21875,0.0,0.0
22,2025-03-02,43.475,356.275,0.608833,0.0,0.830890,0.15625,0.0,0.0
23,2025-03-02,43.475,356.325,1.157715,0.0,0.834066,0.31250,0.0,0.0
